# REHAB24-6 MediaPipe skeleton features (Colab CPU)

This notebook extracts skeleton features from the RGB videos using MediaPipe Pose (BlazePose), generating the `.npz` files for the correctly/incorrectly classified repetition evaluation. MediaPipe is CPU-bound; running it in Colab avoids tying up your local machine for hours.

## Before you run
Upload the repo folder (including `data/REHAB24-6/`) to your Google Drive, e.g. `MyDrive/x-coach/`. The `data/REHAB24-6/processed/manifest.csv` + `splits/` must be present (already built locally). Then run the cells top to bottom. The extractor is **resumable** (it skips reps whose `.npz` already exists), so if Colab disconnects you can re-run the same cells and it continues.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install mediapipe
!pip -q install mediapipe opencv-python-headless scipy numpy

In [ ]:
# 3. Paths. REPO_ON_DRIVE = where you uploaded the repo (with data/REHAB24-6 inside).
import sys, shutil, time
from pathlib import Path

REPO_ON_DRIVE = Path('/content/drive/MyDrive/x-coach')          # <-- adjust if you uploaded elsewhere
LOCAL_DATA   = Path('/content/REHAB24-6')                        # fast local copy for video decode

assert REPO_ON_DRIVE.exists(), f'Repo not found on Drive: {REPO_ON_DRIVE}'
sys.path.insert(0, str(REPO_ON_DRIVE))                            # so `import src...` works

# Copy data/REHAB24-6 from Drive to local disk (Drive FUSE is slow for heavy video IO).
src_data = REPO_ON_DRIVE / 'data' / 'REHAB24-6'
assert (src_data / 'processed' / 'manifest.csv').exists(), 'manifest.csv missing — build it locally first.'
if not LOCAL_DATA.exists():
    print('Copying videos to local disk (one-time, a few minutes for ~7.8GB)...')
    t = time.time(); shutil.copytree(src_data, LOCAL_DATA)
    print(f'  copied in {time.time()-t:.0f}s')
else:
    print('Local copy already present:', LOCAL_DATA)
print('videos:', len(list(LOCAL_DATA.rglob('*.mp4'))))

In [ ]:
# 4. Smoke test: extract ONE video first to validate the runtime end-to-end.
from src.rehab24.mediapipe_skeleton_features import extract_features_for_manifest

MANIFEST   = LOCAL_DATA / 'processed' / 'manifest.csv'
OUTPUT_DIR = LOCAL_DATA / 'processed' / 'mediapipe_skeleton_features'

# We use num_workers=2 because Colab standard CPU has 2 cores.
written = extract_features_for_manifest(
    data_root=LOCAL_DATA, manifest_path=MANIFEST, output_dir=OUTPUT_DIR,
    model_complexity=2, frame_stride=1, num_workers=2, video_limit=1
)
print('smoke-test reps written:', written)

import numpy as np
sample = next(OUTPUT_DIR.rglob('*.npz'))
with np.load(sample) as d:
    f = d['video_feature']
print('sample', sample.name, '| feature_dim', f.shape[0], '| finite', bool(np.isfinite(f).all()))

In [ ]:
# 5. Full extraction (all 130 videos). Resumable: re-run this cell after any disconnect.
written = extract_features_for_manifest(
    data_root=LOCAL_DATA, manifest_path=MANIFEST, output_dir=OUTPUT_DIR,
    model_complexity=2, frame_stride=1, num_workers=2
)
print('reps written this run:', written)
n = len(list(OUTPUT_DIR.rglob('*.npz')))
print('total .npz now:', n, '(expect 2144)')

In [ ]:
# 6. Verify all reps are present and finite
import numpy as np
paths = list(OUTPUT_DIR.rglob('*.npz'))
bad = [p.name for p in paths if not np.isfinite(np.load(p)['video_feature']).all()]
print(f'total={len(paths)}  non-finite={len(bad)}')
print('OK' if len(paths) == 2144 and not bad else f'CHECK: {bad[:5]}')

In [ ]:
# 7. Copy the feature dir back to Drive so you can pull it locally for LOSO.
import subprocess
dst = REPO_ON_DRIVE / 'data' / 'REHAB24-6' / 'processed' / 'mediapipe_skeleton_features'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(OUTPUT_DIR, dst)
size = subprocess.run(['du','-sh',str(dst)], capture_output=True, text=True).stdout.strip()
print('copied features to Drive:', size)

## Back on your local machine
Pull the feature dir from Drive into the repo, then train + cross-validate:
```bash
# place it at: data/REHAB24-6/processed/mediapipe_skeleton_features/
source .venv/bin/activate

# single fixed-split run
python scripts/rehab24/train_correctness_classifier.py \
  --feature-dir data/REHAB24-6/processed/mediapipe_skeleton_features

# LOSO mean±std
python scripts/rehab24/loso_cross_validation.py \
  --feature-dir data/REHAB24-6/processed/mediapipe_skeleton_features \
  --summary-output data/REHAB24-6/processed/correctness_loso_mediapipe.json
```